In [ ]:
"""
Multi-Scale Random Fourier Features Reservoir Computing (RFF-RC)
for the Rulkov map — grid search aligned with Laha (2025).

Paper reference: "Reservoir Computing via Multi-Scale Random Fourier Features
for Forecasting Fast-Slow Dynamical Systems", S.K. Laha.

Key design choices (see header comments in the accompanying explanation):
  - RFF frequencies sampled ~ N(0, 1/sigma^2), phases ~ U[0, 2*pi)
  - Per-variable bandwidth sigma_x (fast), sigma_y (slow)
  - Per-variable RFF block dimension D_x, D_y, concatenated
  - Ridge readout on concatenated feature matrix
  - Closed-loop prediction by delay-window shift + re-featurisation each step

Hyperparameter priority (based on the paper's mechanism and RC practice):
  HIGHEST:  sigma_x, sigma_y    (mechanism of the multi-scale claim)
  HIGH:     ridge_alpha         (closed-loop stability)
  MEDIUM:   lag                 (Rulkov is 2D Markov so small lag suffices)
  LOW:      D_x, D_y            (returns diminish past ~1000; fixed here)
"""

import numpy as np
import matplotlib.pyplot as plt
import itertools
import time

from sklearn.linear_model import Ridge

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))
from lib.utils.reservoirpy import (
    fit_scaler, transform_array, inverse_transform_array
)


# ==========================================================
# FIXED PARAMETERS
# ==========================================================
# These are set to the most reasonable values given the paper and RC theory.
# Rationale for each:
#
#  train_len / test_len : matches the paper's ~5000/1000 Rulkov split
#                         (inferred from Fig. 1).
#  normalization        : minmax11 for numerical stability of RFF cosines.
#                         NOTE: the paper's raw sigma_x=0.1, sigma_y=10.0
#                         correspond to *raw* Rulkov signal scales. Once we
#                         normalize, the bandwidth grid below retunes for
#                         signals in [-1, 1].
#  D_x, D_y             : 1000 features per variable. RFF is a Monte Carlo
#                         kernel approximation: variance ~ 1/D, so beyond
#                         ~500-1000 the gains saturate. The paper does not
#                         specify D; 1000 is a standard default.
#  base_seed            : for reproducibility of the grid (top-K are
#                         re-evaluated over multiple seeds).
FIXED = {
    "train_len":     4000,
    "test_start":    4000,
    "test_len":      1000,
    "normalization": "minmax11",
    "D_x":           1000,
    "D_y":           1000,
    "base_seed":     42,
    "n_final_seeds": 5,   # multi-seed re-evaluation of best configs
    "top_k":         10,  # how many best configs to re-evaluate
}


# ==========================================================
# GRID — FOUR PARAMETERS THAT ACTUALLY MATTER
# ==========================================================
# sigma_x, sigma_y  [HIGHEST PRIORITY]
#    On minmax11-scaled data (both vars in [-1, 1]):
#    - fast x should get a narrower kernel (small sigma -> high-frequency RFF
#      features that resolve spike structure)
#    - slow y should get a broader kernel (large sigma -> low-frequency RFF
#      features that track slow drift)
#    The grid spans ~3 decades for sigma_x and includes values well below 1
#    to resolve sharp voltage excursions.
#
# ridge_alpha  [HIGH PRIORITY]
#    Controls bias-variance trade-off of the readout. Closed-loop rollouts
#    are *very* sensitive to under-regularization: a readout that overfits
#    training residuals will amplify errors in autoregression. Grid spans
#    1e-10 to 1e-2 log-uniformly.
#
# lag  [MEDIUM PRIORITY]
#    Rulkov is a 2D Markov map: x_{n+1} depends on x_n and y_n only. So
#    lag=2 is theoretically sufficient. Larger lags add robustness against
#    noise/finite-D RFF approximation but also add useless dimensions.
#    Grid: {2, 4, 8, 16}.
GRID = {
    "lag":         [2, 4, 8, 16],
    "sigma_x":     [0.03, 0.1, 0.3, 1.0, 3.0],
    "sigma_y":     [0.1, 0.3, 1.0, 3.0, 10.0, 30.0],
    "ridge_alpha": [1e-10, 1e-8, 1e-6, 1e-4, 1e-2],
}

# Optional: also include a single-scale baseline grid for comparison.
# Single global bandwidth applied to the full concatenated delay vector.
GRID_SINGLE = {
    "lag":          [2, 4, 8, 16],
    "sigma_single": [0.1, 0.3, 1.0, 3.0, 10.0],
    "ridge_alpha":  [1e-10, 1e-8, 1e-6, 1e-4, 1e-2],
    "D_total":      [2000],  # 2*D to match multi-scale capacity
}

COMPARE_SINGLE_SCALE = True


# ==========================================================
# DATA LOADING
# ==========================================================
dataset = np.loadtxt('../../../data/chaotic_data/rulkov_map.csv', delimiter=',')
if dataset.ndim == 1:
    dataset = dataset.reshape(-1, 1)
if dataset.shape[1] >= 2:
    dataset = dataset[:, :2]

n_vars_data = dataset.shape[1]
if n_vars_data < 2:
    raise RuntimeError(
        "This script is intended for the full 2-variable Rulkov signal. "
        "Please provide a CSV with columns [x, y]."
    )

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(dataset[:, 0], linewidth=0.8, label='x (fast)')
ax.plot(dataset[:, 1], linewidth=0.8, label='y (slow)')
ax.legend(); ax.set_xlabel("Step"); ax.set_ylabel("State")
ax.set_title("Rulkov map training signal")
plt.tight_layout()
plt.show()


# ==========================================================
# CORE FUNCTIONS
# ==========================================================
def build_delay_embedding_multivar(data, lag):
    """Concatenated per-variable delay vectors, most-recent-first within blocks."""
    T, n_vars = data.shape
    X, Y = [], []
    for t in range(lag, T):
        row = [data[t - lag:t, j][::-1] for j in range(n_vars)]
        X.append(np.concatenate(row))
        Y.append(data[t])
    return np.asarray(X), np.asarray(Y)


def rff_map(X, W, b):
    """RFF feature map: sqrt(2/D) * cos(X W + b). Approximates Gaussian RBF kernel."""
    D = W.shape[1]
    return np.sqrt(2.0 / D) * np.cos(X @ W + b)


class MultiScaleRFFReservoir:
    """Per-variable RFF blocks with their own bandwidth; concatenated."""
    def __init__(self, lag, n_vars, D_per_var, sigmas, rng_seed=42):
        self.lag = lag
        self.n_vars = n_vars
        rng = np.random.RandomState(rng_seed)
        self.W_blocks = []
        self.b_blocks = []
        for j in range(n_vars):
            D = int(D_per_var[j])
            sigma = float(sigmas[j])
            # Gaussian RBF RFF: W ~ N(0, sigma^{-2} I), b ~ U[0, 2*pi)
            self.W_blocks.append(
                rng.normal(loc=0.0, scale=1.0 / sigma, size=(lag, D))
            )
            self.b_blocks.append(rng.uniform(0.0, 2.0 * np.pi, size=D))

    def transform(self, X_delay):
        feats = []
        for j in range(self.n_vars):
            start = j * self.lag
            end   = (j + 1) * self.lag
            feats.append(rff_map(X_delay[:, start:end],
                                 self.W_blocks[j], self.b_blocks[j]))
        return np.hstack(feats)


class SingleScaleRFFReservoir:
    """One global bandwidth on the full concatenated delay vector."""
    def __init__(self, input_dim, D_total, sigma, rng_seed=42):
        rng = np.random.RandomState(rng_seed)
        self.W = rng.normal(loc=0.0, scale=1.0 / sigma,
                            size=(input_dim, D_total))
        self.b = rng.uniform(0.0, 2.0 * np.pi, size=D_total)

    def transform(self, X_delay):
        return rff_map(X_delay, self.W, self.b)


# ==========================================================
# METRICS
# ==========================================================
def nrmse_per_var(y_true, y_pred):
    """Per-variable NRMSE with std-based normalization."""
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))
    denom = np.std(y_true, axis=0)
    denom = np.where(denom == 0, 1.0, denom)
    return rmse / denom


def valid_prediction_time(y_true, y_pred, threshold=0.4):
    """
    Number of initial steps during which normalized per-variable error
    stays below `threshold`. This is more informative than mean NRMSE for
    chaotic closed-loop where NRMSE saturates near sqrt(2) after divergence.
    """
    scale = np.std(y_true, axis=0)
    scale = np.where(scale == 0, 1.0, scale)
    err = np.abs(y_true - y_pred) / scale          # (T, n_vars)
    max_err = err.max(axis=1)                       # worst variable at each step
    bad = np.where(max_err > threshold)[0]
    return int(bad[0]) if len(bad) > 0 else len(y_true)


def is_diverged(y_pred):
    """Detect numerical blowup or NaN."""
    return (not np.all(np.isfinite(y_pred))) or np.max(np.abs(y_pred)) > 1e4


def update_delay_vector(delay_vec, pred, n_vars, lag):
    """Shift each per-variable block by one; insert new prediction at front."""
    out = delay_vec.copy()
    for j in range(n_vars):
        start = j * lag
        end   = (j + 1) * lag
        block = out[start:end]
        block[1:] = block[:-1]
        block[0]  = pred[j]
        out[start:end] = block
    return out


# ==========================================================
# EVALUATION
# ==========================================================
def evaluate_rff_rc(
    data,
    mode,                # "multiscale" | "singlescale"
    lag,
    ridge_alpha,
    # multi-scale params
    sigma_x=None, sigma_y=None, D_x=None, D_y=None,
    # single-scale params
    sigma_single=None, D_total=None,
    # common
    normalization="minmax11",
    train_len=4000, test_start=4000, test_len=1000,
    rng_seed=42,
    vpt_threshold=0.4,
):
    try:
        X_raw, Y_raw = build_delay_embedding_multivar(data, lag)
        X_train_raw = X_raw[:train_len]
        Y_train_raw = Y_raw[:train_len]
        X_test_raw  = X_raw[test_start:test_start + test_len]
        Y_test_raw  = Y_raw[test_start:test_start + test_len]
        if len(X_train_raw) == 0 or len(X_test_raw) == 0:
            return None

        n_vars = Y_raw.shape[1]

        x_scaler = fit_scaler(X_train_raw, method=normalization)
        y_scaler = fit_scaler(Y_train_raw, method=normalization)
        X_train = transform_array(X_train_raw, x_scaler)
        Y_train = transform_array(Y_train_raw, y_scaler)
        X_test  = transform_array(X_test_raw,  x_scaler)
        Y_test  = transform_array(Y_test_raw,  y_scaler)

        if mode == "multiscale":
            reservoir = MultiScaleRFFReservoir(
                lag=lag, n_vars=n_vars,
                D_per_var=[D_x, D_y],
                sigmas=[sigma_x, sigma_y],
                rng_seed=rng_seed,
            )
        elif mode == "singlescale":
            reservoir = SingleScaleRFFReservoir(
                input_dim=X_train.shape[1],
                D_total=D_total,
                sigma=sigma_single,
                rng_seed=rng_seed,
            )
        else:
            raise ValueError(mode)

        F_train = reservoir.transform(X_train)
        readout = Ridge(alpha=ridge_alpha)
        readout.fit(F_train, Y_train)

        # --- one-step ---
        F_test = reservoir.transform(X_test)
        Y_pred_1_s = readout.predict(F_test)
        Y_pred_1 = inverse_transform_array(Y_pred_1_s, y_scaler)
        Y_true_1 = inverse_transform_array(Y_test, y_scaler)
        nrmse_1 = nrmse_per_var(Y_true_1, Y_pred_1)

        # --- closed-loop ---
        pred_len = len(X_test)
        current = X_test[0].copy()
        Y_cl_s = np.zeros((pred_len, n_vars))
        for k in range(pred_len):
            feat = reservoir.transform(current.reshape(1, -1))
            pred = readout.predict(feat)[0]
            Y_cl_s[k] = pred
            current = update_delay_vector(current, pred, n_vars, lag)

        Y_pred_cl = inverse_transform_array(Y_cl_s, y_scaler)
        Y_true_cl = inverse_transform_array(Y_test[:pred_len], y_scaler)

        diverged = is_diverged(Y_pred_cl)
        if diverged:
            # Clip for metric computation but flag it
            Y_pred_cl = np.nan_to_num(Y_pred_cl, nan=0.0, posinf=1e4, neginf=-1e4)

        nrmse_cl = nrmse_per_var(Y_true_cl, Y_pred_cl)
        vpt = valid_prediction_time(Y_true_cl, Y_pred_cl, threshold=vpt_threshold)

        return {
            "mode": mode,
            "lag": lag,
            "ridge_alpha": ridge_alpha,
            "sigma_x": sigma_x, "sigma_y": sigma_y,
            "D_x": D_x, "D_y": D_y,
            "sigma_single": sigma_single, "D_total": D_total,
            "seed": rng_seed,
            "one_step_nrmse_vec":  nrmse_1,
            "one_step_nrmse_mean": float(np.mean(nrmse_1)),
            "closed_nrmse_vec":    nrmse_cl,
            "closed_nrmse_mean":   float(np.mean(nrmse_cl)),
            "vpt":                 vpt,
            "diverged":            bool(diverged),
            "Y_true_cl":           Y_true_cl,
            "Y_pred_cl":           Y_pred_cl,
            "Y_true_1":            Y_true_1,
            "Y_pred_1":            Y_pred_1,
            "n_vars":              n_vars,
        }
    except Exception as e:
        print(f"ERROR: {e}")
        return None


# ==========================================================
# GRID SEARCH
# ==========================================================
def make_combos(grid):
    keys = list(grid.keys())
    return keys, list(itertools.product(*[grid[k] for k in keys]))


def run_multiscale_grid(data):
    keys, combos = make_combos(GRID)
    print(f"\nMulti-scale grid: {len(combos)} configs")
    results = []
    t0 = time.time()
    for i, combo in enumerate(combos, 1):
        params = dict(zip(keys, combo))
        r = evaluate_rff_rc(
            data=data, mode="multiscale",
            lag=params["lag"],
            ridge_alpha=params["ridge_alpha"],
            sigma_x=params["sigma_x"], sigma_y=params["sigma_y"],
            D_x=FIXED["D_x"], D_y=FIXED["D_y"],
            normalization=FIXED["normalization"],
            train_len=FIXED["train_len"],
            test_start=FIXED["test_start"],
            test_len=FIXED["test_len"],
            rng_seed=FIXED["base_seed"],
        )
        if r is not None:
            results.append(r)
        if i % 25 == 0 or i == len(combos):
            elapsed = time.time() - t0
            best_so_far = min((x["closed_nrmse_mean"] for x in results
                               if not x["diverged"]), default=np.inf)
            best_vpt   = max((x["vpt"] for x in results), default=0)
            print(f"  [multi {i:4d}/{len(combos)}]  "
                  f"elapsed={elapsed:6.1f}s  "
                  f"best_closed_nrmse={best_so_far:.4e}  "
                  f"best_vpt={best_vpt}")
    return results


def run_singlescale_grid(data):
    keys, combos = make_combos(GRID_SINGLE)
    print(f"\nSingle-scale grid: {len(combos)} configs")
    results = []
    t0 = time.time()
    for i, combo in enumerate(combos, 1):
        params = dict(zip(keys, combo))
        r = evaluate_rff_rc(
            data=data, mode="singlescale",
            lag=params["lag"],
            ridge_alpha=params["ridge_alpha"],
            sigma_single=params["sigma_single"],
            D_total=params["D_total"],
            normalization=FIXED["normalization"],
            train_len=FIXED["train_len"],
            test_start=FIXED["test_start"],
            test_len=FIXED["test_len"],
            rng_seed=FIXED["base_seed"],
        )
        if r is not None:
            results.append(r)
        if i % 25 == 0 or i == len(combos):
            elapsed = time.time() - t0
            best = min((x["closed_nrmse_mean"] for x in results
                        if not x["diverged"]), default=np.inf)
            best_vpt = max((x["vpt"] for x in results), default=0)
            print(f"  [single {i:4d}/{len(combos)}]  "
                  f"elapsed={elapsed:6.1f}s  "
                  f"best_closed_nrmse={best:.4e}  "
                  f"best_vpt={best_vpt}")
    return results


def rank_results(results, primary="vpt"):
    """
    Primary ranking key. For chaotic closed-loop, VPT is more informative
    than mean NRMSE (which saturates). Ties broken by closed_nrmse_mean.
    Diverged runs pushed to the bottom.
    """
    def key(r):
        div_penalty = 1 if r["diverged"] else 0
        if primary == "vpt":
            return (div_penalty, -r["vpt"], r["closed_nrmse_mean"])
        else:
            return (div_penalty, r["closed_nrmse_mean"], -r["vpt"])
    return sorted(results, key=key)


def multi_seed_eval(best_config, data, n_seeds=5):
    """Re-evaluate a config across multiple RFF seeds; return stats."""
    scores = []
    vpts = []
    for s in range(FIXED["base_seed"], FIXED["base_seed"] + n_seeds):
        r = evaluate_rff_rc(
            data=data,
            mode=best_config["mode"],
            lag=best_config["lag"],
            ridge_alpha=best_config["ridge_alpha"],
            sigma_x=best_config.get("sigma_x"),
            sigma_y=best_config.get("sigma_y"),
            D_x=FIXED["D_x"], D_y=FIXED["D_y"],
            sigma_single=best_config.get("sigma_single"),
            D_total=best_config.get("D_total"),
            normalization=FIXED["normalization"],
            train_len=FIXED["train_len"],
            test_start=FIXED["test_start"],
            test_len=FIXED["test_len"],
            rng_seed=s,
        )
        if r is None:
            continue
        scores.append(r["closed_nrmse_mean"])
        vpts.append(r["vpt"])
    return {
        "closed_nrmse_mean_median": float(np.median(scores)) if scores else np.nan,
        "closed_nrmse_mean_iqr":    float(np.subtract(*np.percentile(scores, [75, 25]))) if scores else np.nan,
        "vpt_median":               float(np.median(vpts)) if vpts else 0,
        "vpt_min":                  int(np.min(vpts)) if vpts else 0,
        "n_seeds_ok":               len(scores),
    }


# ==========================================================
# MAIN
# ==========================================================
all_results = []
all_results += run_multiscale_grid(dataset)
if COMPARE_SINGLE_SCALE:
    all_results += run_singlescale_grid(dataset)

# Rank primarily by VPT (steps-until-divergence), tiebreak by NRMSE.
ranked = rank_results(all_results, primary="vpt")


# ==========================================================
# REPORT
# ==========================================================
print("\n" + "=" * 100)
print(f"TOP {FIXED['top_k']} CONFIGURATIONS  (ranked by valid prediction time, then NRMSE)")
print("=" * 100)

def fmt_row(rank, r):
    base = (f"{rank:2d}. {r['mode']:11s} | "
            f"lag={r['lag']:2d} | "
            f"ridge={r['ridge_alpha']:.0e} | "
            f"vpt={r['vpt']:4d} | "
            f"closed_nrmse={r['closed_nrmse_mean']:.3e} | "
            f"onestep={r['one_step_nrmse_mean']:.3e}")
    if r['mode'] == "multiscale":
        extra = f" | sigma_x={r['sigma_x']:6.3f} sigma_y={r['sigma_y']:6.3f}"
    else:
        extra = f" | sigma={r['sigma_single']:6.3f} D_total={r['D_total']}"
    diverged = "  [DIVERGED]" if r['diverged'] else ""
    return base + extra + diverged

for rank, r in enumerate(ranked[:FIXED["top_k"]], 1):
    print(fmt_row(rank, r))


# ==========================================================
# MULTI-SEED RE-EVALUATION OF TOP K
# ==========================================================
print("\n" + "=" * 100)
print(f"MULTI-SEED RE-EVALUATION of top {FIXED['top_k']} configs "
      f"({FIXED['n_final_seeds']} seeds each)")
print("=" * 100)

seed_eval = []
for rank, r in enumerate(ranked[:FIXED["top_k"]], 1):
    stats = multi_seed_eval(r, dataset, n_seeds=FIXED["n_final_seeds"])
    seed_eval.append((r, stats))
    tag = (f"sigma_x={r['sigma_x']:.3f},sigma_y={r['sigma_y']:.3f}"
           if r['mode'] == "multiscale"
           else f"sigma={r['sigma_single']:.3f}")
    print(f"{rank:2d}. {r['mode']:11s} lag={r['lag']} ridge={r['ridge_alpha']:.0e} "
          f"{tag} | "
          f"vpt median={stats['vpt_median']:.0f} (min={stats['vpt_min']}) | "
          f"closed_nrmse median={stats['closed_nrmse_mean_median']:.3e} "
          f"(IQR={stats['closed_nrmse_mean_iqr']:.3e})")

# Best under multi-seed: highest median VPT, tiebreak lowest median NRMSE.
seed_eval.sort(
    key=lambda t: (-t[1]["vpt_median"], t[1]["closed_nrmse_mean_median"])
)
best_r, best_stats = seed_eval[0]

print("\n" + "=" * 100)
print("FINAL BEST CONFIGURATION (multi-seed robust)")
print("=" * 100)
print(f"mode            : {best_r['mode']}")
print(f"lag             : {best_r['lag']}")
print(f"ridge_alpha     : {best_r['ridge_alpha']:.0e}")
if best_r['mode'] == "multiscale":
    print(f"sigma_x         : {best_r['sigma_x']}")
    print(f"sigma_y         : {best_r['sigma_y']}")
    print(f"D_x, D_y        : {FIXED['D_x']}, {FIXED['D_y']}")
else:
    print(f"sigma_single    : {best_r['sigma_single']}")
    print(f"D_total         : {best_r['D_total']}")
print(f"VPT median      : {best_stats['vpt_median']:.0f} "
      f"(min across seeds: {best_stats['vpt_min']})")
print(f"closed NRMSE med: {best_stats['closed_nrmse_mean_median']:.4e}")
print(f"per-var one-step NRMSE (seed {FIXED['base_seed']}): "
      f"{best_r['one_step_nrmse_vec']}")
print(f"per-var closed  NRMSE (seed {FIXED['base_seed']}): "
      f"{best_r['closed_nrmse_vec']}")


# ==========================================================
# PLOTS — best closed-loop and one-step
# ==========================================================
Y_true_cl = best_r["Y_true_cl"]
Y_pred_cl = best_r["Y_pred_cl"]
Y_true_1  = best_r["Y_true_1"]
Y_pred_1  = best_r["Y_pred_1"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(Y_true_cl[:, 0], c="green", label="Ground truth x", lw=1.0)
axes[0].plot(Y_pred_cl[:, 0], "--", c="red", label="Predicted x", lw=1.0)
axes[0].axvline(best_stats["vpt_median"], color="k", ls=":", alpha=0.5,
                label=f"VPT={best_stats['vpt_median']:.0f}")
axes[0].set_ylabel("x (fast)"); axes[0].legend(loc="upper right")

axes[1].plot(Y_true_cl[:, 1], c="green", label="Ground truth y", lw=1.0)
axes[1].plot(Y_pred_cl[:, 1], "--", c="red", label="Predicted y", lw=1.0)
axes[1].axvline(best_stats["vpt_median"], color="k", ls=":", alpha=0.5)
axes[1].set_ylabel("y (slow)"); axes[1].set_xlabel("Step")
axes[1].legend(loc="upper right")

title = (f"Best RFF-RC closed-loop ({best_r['mode']}) | "
         f"lag={best_r['lag']} ridge={best_r['ridge_alpha']:.0e}")
if best_r['mode'] == "multiscale":
    title += f" sigma_x={best_r['sigma_x']} sigma_y={best_r['sigma_y']}"
else:
    title += f" sigma={best_r['sigma_single']}"
fig.suptitle(title)
plt.tight_layout()
plt.show()


# Log-scale per-step error — like paper's Fig. 3
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
err = np.abs(Y_true_cl - Y_pred_cl) + 1e-12
axes[0].semilogy(err[:, 0], c="red", lw=0.8)
axes[0].set_ylabel("|err| x"); axes[0].grid(True, which="both", alpha=0.3)
axes[1].semilogy(err[:, 1], c="red", lw=0.8)
axes[1].set_ylabel("|err| y"); axes[1].set_xlabel("Step")
axes[1].grid(True, which="both", alpha=0.3)
fig.suptitle("Closed-loop absolute error (log scale) — best config")
plt.tight_layout()
plt.show()